In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =============================================
# CELL 1: Cài đặt các thư viện cần thiết
# =============================================
!pip install -q transformers datasets sentencepiece sacrebleu evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [3]:
# =============================================
# CELL 2: Import các modules cần thiết
# =============================================
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed
)
import evaluate

# Cố định seed để kết quả tái lập được
set_seed(42)

In [4]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("ura-hcmut/PhoMT") 
train_dataset = dataset["train"].select(range(133317))
valid_dataset = dataset["validation"].select(range(1268))

print(f"Train size : {len(train_dataset)}")
print(f"Valid size : {len(valid_dataset)}")
print(f"Sample     : {train_dataset[0]}")

README.md:   0%|          | 0.00/653 [00:00<?, ?B/s]

PhoMT_training.csv:   0%|          | 0.00/525M [00:00<?, ?B/s]

PhoMT_validation.csv: 0.00B [00:00, ?B/s]

PhoMT_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2977999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/18720 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19151 [00:00<?, ? examples/s]

Train size : 133317
Valid size : 1268
Sample     : {'en': 'It begins with a countdown.', 'vi': 'Câu chuyện bắt đầu với buổi lễ đếm ngược.'}


In [5]:
# =============================================
# CELL 4: Load model và tokenizer
# =============================================
# Helsinki-NLP/opus-mt-en-vi là mô hình MarianMT đã được pre-train
# cho cặp ngôn ngữ Anh → Việt. Đây là kiến trúc Transformer Encoder-Decoder
# chuyên biệt cho dịch máy, KHÔNG phải mô hình T5 hay GPT.
model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model loaded  : {model_name}")
print(f"Vocab size    : {tokenizer.vocab_size}")
print(f"Model params  : {model.num_parameters():,}") 

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded  : Helsinki-NLP/opus-mt-en-vi
Vocab size    : 53685
Model params  : 127,122,944


In [6]:
# =============================================
# CELL 5: Tiền xử lý dữ liệu (Tokenization)
# =============================================
# LƯU Ý QUAN TRỌNG:
# MarianMT (opus-mt) KHÔNG dùng prefix như T5/mT5.
# Việc thêm prefix "translate English to Vietnamese:" vào MarianMT
# là SAI vì model không được huấn luyện để nhận prefix đó,
# dẫn đến giảm chất lượng dịch.

MAX_LENGTH = 128

def preprocess(example):
    # Tokenize câu nguồn (tiếng Anh) — KHÔNG thêm prefix
    model_inputs = tokenizer(
        example["en"],
        max_length=MAX_LENGTH,
        truncation=True
    )

    # Tokenize câu đích (tiếng Việt) dùng text_target
    # text_target đảm bảo tokenizer dùng đúng vocabulary phía target
    labels = tokenizer(
        text_target=example["vi"],
        max_length=MAX_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs 

In [7]:
# =============================================
# CELL 6: Áp dụng preprocessing lên toàn bộ dataset
# =============================================
tokenized_train = train_dataset.map(
    preprocess,
    batched=True,                        # Xử lý theo batch để nhanh hơn
    remove_columns=train_dataset.column_names  # Bỏ cột gốc "en", "vi"
)

tokenized_valid = valid_dataset.map(
    preprocess,
    batched=True,
    remove_columns=valid_dataset.column_names
)

print("Tokenization hoàn tất!")
print(f"Train features: {tokenized_train.features}") 

Map:   0%|          | 0/133317 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

Tokenization hoàn tất!
Train features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [8]:
# =============================================
# CELL 7: Tạo Data Collator
# =============================================
# DataCollatorForSeq2Seq tự động padding các sequence trong cùng batch
# về cùng độ dài, và thay padding trong labels bằng -100
# (để loss function bỏ qua các vị trí padding khi tính loss)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
) 

In [9]:
# =============================================
# CELL 8: Định nghĩa hàm tính BLEU score
# =============================================
# BLEU (Bilingual Evaluation Understudy) là thước đo tiêu chuẩn
# để đánh giá chất lượng dịch máy, đo mức độ trùng khớp n-gram
# giữa bản dịch của máy và bản dịch tham chiếu của con người.
# Điểm BLEU từ 0-100, càng cao càng tốt.

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Decode các token dự đoán thành text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Thay -100 (padding trong labels) bằng pad_token_id để decode được
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Loại bỏ khoảng trắng thừa
    decoded_preds  = [p.strip() for p in decoded_preds]
    # sacrebleu yêu cầu mỗi reference là một list (hỗ trợ nhiều reference)
    decoded_labels = [[l.strip()] for l in decoded_labels]

    result = bleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    # sacrebleu trả về score từ 0-100 trực tiếp
    return {"bleu": round(result["score"], 4)} 

In [10]:
import transformers
print(transformers.__version__)

# Kiểm tra các tham số hợp lệ của Seq2SeqTrainingArguments
import inspect
params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
print(list(params.keys()))

5.0.0
['self', 'output_dir', 'do_train', 'do_eval', 'do_predict', 'eval_strategy', 'prediction_loss_only', 'per_device_train_batch_size', 'per_device_eval_batch_size', 'gradient_accumulation_steps', 'eval_accumulation_steps', 'eval_delay', 'torch_empty_cache_steps', 'learning_rate', 'weight_decay', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'max_grad_norm', 'num_train_epochs', 'max_steps', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_ratio', 'warmup_steps', 'log_level', 'log_level_replica', 'log_on_each_node', 'logging_dir', 'logging_strategy', 'logging_first_step', 'logging_steps', 'logging_nan_inf_filter', 'save_strategy', 'save_steps', 'save_total_limit', 'enable_jit_checkpoint', 'save_on_each_node', 'save_only_model', 'restore_callback_states_from_checkpoint', 'use_cpu', 'seed', 'data_seed', 'bf16', 'fp16', 'bf16_full_eval', 'fp16_full_eval', 'tf32', 'local_rank', 'ddp_backend', 'debug', 'dataloader_drop_last', 'eval_steps', 'dataloader_num_workers', 'dataloader_prefetch_fa

In [11]:
!pip install -q --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 74.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.3 MB/s eta 0:00:0000:01


In [12]:
# =============================================
# CELL 9: Cấu hình Training Arguments
# =============================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./envi-model",

    # --- Evaluation & Saving ---
    eval_strategy="epoch",         
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,

    # --- Batch size ---
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # --- Optimizer ---
    learning_rate=2e-5,
    warmup_ratio=0.1,

    # --- Epochs ---
    num_train_epochs=5,

    # --- Generation ---
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=5,

    # --- Logging & Misc ---
    logging_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

print("✅ Training args OK!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training args OK!


In [13]:
# =============================================
# CELL 10: Khởi tạo Trainer và bắt đầu train
# =============================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics   # Tự động tính BLEU sau mỗi epoch
)

# Bắt đầu fine-tuning
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Bleu
1,3.124730,1.507276,34.247600
2,2.899056,1.519996,33.660300
3,2.792492,1.523745,33.655300
4,2.667227,1.530776,33.011300
5,2.541591,1.533027,33.511500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=41665, training_loss=2.8227398248761983, metrics={'train_runtime': 9654.0855, 'train_samples_per_second': 69.047, 'train_steps_per_second': 4.316, 'total_flos': 9615537366958080.0, 'train_loss': 2.8227398248761983, 'epoch': 5.0})

In [14]:
# =============================================
# CELL 11: Inference - Dịch thử một số câu
# =============================================
# Dùng batch inference thay vì loop từng câu để nhanh hơn

model.eval()  # Chuyển sang chế độ evaluation (tắt dropout)

test_sentences = [
    "I love machine learning.",
    "This is a beautiful day.",
    "We will meet tomorrow.",
    "Artificial intelligence is changing the world.",
    "Machine learning is interesting."
]

# Tokenize cả batch cùng lúc (KHÔNG thêm prefix với MarianMT)
inputs = tokenizer(
    test_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5,              # Beam search để chọn bản dịch tốt nhất
        no_repeat_ngram_size=3,   # Tránh lặp lại n-gram
        early_stopping=True       # Dừng sớm khi tất cả beam đã kết thúc
    )

decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
for en, vi in zip(test_sentences, decoded):
    print(f"EN: {en}")
    print(f"VI: {vi}")
    print("-" * 50)

EN: I love machine learning.
VI: Tôi yêu việc học máy móc.
--------------------------------------------------
EN: This is a beautiful day.
VI: Đây là một ngày đẹp trời.
--------------------------------------------------
EN: We will meet tomorrow.
VI: Chúng ta sẽ gặp nhau vào ngày mai.
--------------------------------------------------
EN: Artificial intelligence is changing the world.
VI: Trí tuệ nhân tạo đang thay đổi thế giới.
--------------------------------------------------
EN: Machine learning is interesting.
VI: Việc học máy rất thú vị.
--------------------------------------------------


In [15]:
# =============================================
# CELL 12: Đánh giá BLEU trên tập Validation
# =============================================
# Dùng batch inference thay vì loop từng câu để tăng tốc độ đáng kể

BATCH_SIZE  = 32
NUM_SAMPLES = 1268  # Toàn bộ tập validation

raw_texts = [valid_dataset[i]["en"] for i in range(NUM_SAMPLES)]
raw_refs  = [valid_dataset[i]["vi"] for i in range(NUM_SAMPLES)]

all_preds = []
model.eval()

for i in range(0, NUM_SAMPLES, BATCH_SIZE):
    batch = raw_texts[i : i + BATCH_SIZE]

    # Tokenize batch (KHÔNG thêm prefix)
    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_preds.extend(decoded)

    if (i // BATCH_SIZE) % 5 == 0:
        print(f"Đã xử lý {min(i + BATCH_SIZE, NUM_SAMPLES)}/{NUM_SAMPLES} câu...")

# Format references: mỗi ref là một list (sacrebleu hỗ trợ nhiều ref)
all_refs_fmt = [[r] for r in raw_refs]

result = bleu_metric.compute(predictions=all_preds, references=all_refs_fmt)
print(f"\n✅ BLEU Score: {result['score']:.2f}")

Đã xử lý 32/1268 câu...
Đã xử lý 192/1268 câu...
Đã xử lý 352/1268 câu...
Đã xử lý 512/1268 câu...
Đã xử lý 672/1268 câu...
Đã xử lý 832/1268 câu...
Đã xử lý 992/1268 câu...
Đã xử lý 1152/1268 câu...

✅ BLEU Score: 34.14


In [16]:
# =============================================
# CELL 13: Lưu model và tokenizer
# =============================================
save_path = "/kaggle/working/envi-model-finalversion1"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model đã được lưu tại: {save_path}")

# Nén lại để download
!zip -r envi-model-finalversion1.zip {save_path}
print("✅ Đã nén thành envi-model.zip") 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model đã được lưu tại: /kaggle/working/envi-model-finalversion1
  adding: kaggle/working/envi-model-finalversion1/ (stored 0%)
  adding: kaggle/working/envi-model-finalversion1/target.spm (deflated 50%)
  adding: kaggle/working/envi-model-finalversion1/model.safetensors (deflated 7%)
  adding: kaggle/working/envi-model-finalversion1/generation_config.json (deflated 59%)
  adding: kaggle/working/envi-model-finalversion1/source.spm (deflated 51%)
  adding: kaggle/working/envi-model-finalversion1/vocab.json (deflated 70%)
  adding: kaggle/working/envi-model-finalversion1/config.json (deflated 64%)
  adding: kaggle/working/envi-model-finalversion1/tokenizer_config.json (deflated 67%)
✅ Đã nén thành envi-model.zip
